In [0]:
%run /Shared/insclm_capstone/NB_00_config_loader.py

[SecretScope(name=' kv-insclm-cap-11'), SecretScope(name='kv-insclm')]

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

print("=" * 55)
print("CLAIM STATUS MERGE")
print("=" * 55)

CLAIM STATUS MERGE


In [0]:
print("\n📥 Loading bronze_claim_status_updates...")
bronze_status = spark.table(
    "bronze_insclm.bronze_claim_status_updates")
print(f"   Rows: {bronze_status.count():,}  (expected 1,600)")

source = (bronze_status
    .withColumn("is_terminal_status",
        F.col("new_status").isin("Approved","Rejected"))
    .withColumn("_silver_loaded_at", F.current_timestamp()))

print("\nis_terminal_status distribution:")
source.groupBy("is_terminal_status","new_status") \
    .count().orderBy("is_terminal_status").show()


📥 Loading bronze_claim_status_updates...
   Rows: 1,600  (expected 1,600)

is_terminal_status distribution:
+------------------+-------------+-----+
|is_terminal_status|   new_status|count|
+------------------+-------------+-----+
|             false|      Pending|  258|
|             false|Investigation|  151|
|              true|     Approved|  856|
|              true|     Rejected|  335|
+------------------+-------------+-----+



In [0]:
TABLE_NAME = "silver_insclm.silver_claim_status_history"

if not spark.catalog.tableExists(TABLE_NAME):
    print(f"\n📥 Table does not exist — initial load...")

    source.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema","true") \
        .saveAsTable(TABLE_NAME)

    count = spark.table(TABLE_NAME).count()
    print(f"✅ Initial load → {count:,} rows  (expected 1,600)")

else:
    print(f"\n📥 Table exists — running MERGE...")
    before = spark.table(TABLE_NAME).count()
    target = DeltaTable.forName(spark, TABLE_NAME)

    (target.alias("t")
        .merge(source.alias("s"),
            "t.status_update_id = s.status_update_id")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute())

    after = spark.table(TABLE_NAME).count()
    print(f"   Before : {before:,}")
    print(f"   After  : {after:,}")
    print(f"   New rows added: {after-before:,}")


📥 Table does not exist — initial load...
✅ Initial load → 1,600 rows  (expected 1,600)


In [0]:
print("\n── Status Distribution ──────────────────────────────")
spark.table(TABLE_NAME) \
    .groupBy("new_status") \
    .count() \
    .orderBy(F.desc("count")) \
    .show()

print("── Terminal Status ──────────────────────────────────")
spark.table(TABLE_NAME) \
    .groupBy("is_terminal_status") \
    .count().show()

print("── Delta History ────────────────────────────────────")
DeltaTable.forName(spark, TABLE_NAME) \
    .history(3) \
    .select("version","timestamp","operation") \
    .show(truncate=False)

total = spark.table(TABLE_NAME).count()
print(f"\nFinal row count: {total:,}  (expected 1,600)")
print("✅ NB_05 complete.")
print("   Next: Run NB_06_gold_tables")


── Status Distribution ──────────────────────────────
+-------------+-----+
|   new_status|count|
+-------------+-----+
|     Approved|  856|
|     Rejected|  335|
|      Pending|  258|
|Investigation|  151|
+-------------+-----+

── Terminal Status ──────────────────────────────────
+------------------+-----+
|is_terminal_status|count|
+------------------+-----+
|              true| 1191|
|             false|  409|
+------------------+-----+

── Delta History ────────────────────────────────────
+-------+-------------------+---------------------------------+
|version|timestamp          |operation                        |
+-------+-------------------+---------------------------------+
|0      |2026-05-21 14:57:39|CREATE OR REPLACE TABLE AS SELECT|
+-------+-------------------+---------------------------------+


Final row count: 1,600  (expected 1,600)
✅ NB_05 complete.
   Next: Run NB_06_gold_tables


[SecretMetadata(key='adls-abfss-base'),
 SecretMetadata(key='adls-account-key'),
 SecretMetadata(key='adls-account-name'),
 SecretMetadata(key='adls-audit-path'),
 SecretMetadata(key='adls-base-url'),
 SecretMetadata(key='adls-bronze-path'),
 SecretMetadata(key='adls-container-name'),
 SecretMetadata(key='adls-gold-path'),
 SecretMetadata(key='adls-raw-path'),
 SecretMetadata(key='adls-rejected-path'),
 SecretMetadata(key='adls-silver-path'),
 SecretMetadata(key='database-workspace-url'),
 SecretMetadata(key='databricks-cluster-id'),
 SecretMetadata(key='databricks-pat'),
 SecretMetadata(key='file-claim-status-updates'),
 SecretMetadata(key='file-claims'),
 SecretMetadata(key='file-customer-master'),
 SecretMetadata(key='file-policy-master'),
 SecretMetadata(key='github-pat'),
 SecretMetadata(key='github-repo-url'),
 SecretMetadata(key='sql-admin-name'),
 SecretMetadata(key='sql-admin-password'),
 SecretMetadata(key='sql-connection-string'),
 SecretMetadata(key='sql-database-name'),
 S

✅ Config loaded from Key Vault successfully.
   ADLS Account  : [REDACTED]
   Container     : [REDACTED]
   ABFSS Base    : [REDACTED]
   RAW path      : [REDACTED][REDACTED]
   BRONZE path   : [REDACTED][REDACTED]
   SILVER path   : [REDACTED][REDACTED]
   GOLD path     : [REDACTED][REDACTED]
   REJECTED path : [REDACTED][REDACTED]
   AUDIT path    : [REDACTED][REDACTED]
   SQL Server    : [REDACTED]
   SQL Database  : [REDACTED]
